# P4 — evaluación temporal sin fuga y con autores aislados

Esta versión conserva la cohorte temporal de P4 v3, pero representa explícitamente en el subgrafo histórico a los autores conocidos que todavía no tienen coautorías históricas: se añaden como **nodos aislados**, sin crear enlaces.

- Train/Val: features calculadas solo con historia 2000–2020; labels en 2021–2022.
- Test: features calculadas solo con historia 2000–2022; labels en 2023–2024.
- El futuro se usa únicamente para asignar `connected`.
- Un autor conocido sin coautorías previas no se elimina: permanece como nodo aislado y representa cold-start estructural.
- No se añaden aristas ficticias ni se usa información del horizonte futuro para construir features.

## Puntos a documentar en el artículo

1. La evaluación temporal separa explícitamente historia de features y horizonte de labels.
2. La ausencia de coautorías históricas se trata como estado estructural legítimo, no como dato faltante que motive excluir al autor.
3. Los autores conocidos pero sin enlaces previos se incorporan como nodos aislados en el grafo histórico.
4. El benchmark principal mantiene balance 1:1, pero se evaluarán también escenarios desbalanceados y hard negatives.
5. Se realizará sensibilidad al umbral de coautoría (1, 2, 3 y 5) y se reportará cobertura estructural/cold-start.


In [16]:
from pathlib import Path
from itertools import combinations
from collections import Counter
import random
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.model_selection import train_test_split
import features as ft

BASE_DIR = Path("C:/Users/Dra. Pilar/UltimaversionProgramasComparacion")
SEED = 41
random.seed(SEED)
np.random.seed(SEED)
MIN_COAUTHORED_PAPERS = 1
# P4 original define connected=1 cuando los dos autores comparten al menos un articulo.
# Los umbrales 2, 3 y 5 se reservan para el analisis de sensibilidad.
NEGATIVE_RATIO = 1
NEGATIVE_MODE = 'random'   # 'random' o 'hard'
VAL_SIZE = 0.20
MAX_POSITIVES_TRAINVAL = None
MAX_POSITIVES_TEST = None
WRITE_LEGACY_ALIASES = False

FILES = {
    'full_2000_2020': BASE_DIR / 'BD_UNAM_Graph_2000_2020.gml',
    'sub_2000_2020':  BASE_DIR / 'BD_UNAM_Sub_Graph_2000_2020.gml',
    'full_2021_2022': BASE_DIR / 'BD_UNAM_Graph_2021_2022.gml',
    'sub_2021_2022':  BASE_DIR / 'BD_UNAM_Sub_Graph_2021_2022.gml',
    'full_2023_2024': BASE_DIR / 'BD_UNAM_Graph_2023_2024.gml',
    'sub_2023_2024':  BASE_DIR / 'BD_UNAM_Sub_Graph_2023_2024.gml',
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan archivos GML:\n' + '\n'.join(missing))

G_2000_2020 = nx.read_gml(FILES['full_2000_2020'])
S_2000_2020 = nx.read_gml(FILES['sub_2000_2020'])
G_2021_2022 = nx.read_gml(FILES['full_2021_2022'])
S_2021_2022 = nx.read_gml(FILES['sub_2021_2022'])
G_2023_2024 = nx.read_gml(FILES['full_2023_2024'])
S_2023_2024 = nx.read_gml(FILES['sub_2023_2024'])

G_HIST_2022 = nx.compose(G_2000_2020, G_2021_2022)
S_HIST_2022 = nx.compose(S_2000_2020, S_2021_2022)

print('Grafos cargados:')
for name, G in [('G_2000_2020', G_2000_2020), ('G_2021_2022', G_2021_2022), ('G_2023_2024', G_2023_2024), ('G_HIST_2022', G_HIST_2022)]:
    print(f'{name}: nodos={G.number_of_nodes():,}, aristas={G.number_of_edges():,}')


def node_type(G, node):
    return str(G.nodes[node].get('tipo', '')).strip().lower()


def is_article_node(G, node):
    return node_type(G, node) == 'articulo'


def is_author_node(G, node):
    return node_type(G, node) == 'autor'


def author_nodes(G):
    # CRITICO: solo nodos marcados explicitamente como autores.
    # El grafo principal tambien contiene keywords y afiliaciones.
    return {n for n in G.nodes if is_author_node(G, n)}

def add_isolated_historical_authors(S_history, G_history):
    """Añade al subgrafo de coautoria los autores conocidos sin crear aristas."""
    S = S_history.copy()
    hist_authors = author_nodes(G_history)
    missing = hist_authors - set(S.nodes())
    S.add_nodes_from(missing)
    return S, missing


def canon_pair(u, v):
    return tuple(sorted((str(u), str(v))))


def coauthorship_counts(G, eligible_authors=None):
    eligible = None if eligible_authors is None else set(eligible_authors)
    counts = Counter()
    for article in G.nodes:
        if not is_article_node(G, article):
            continue
        authors = [
            a for a in G.neighbors(article)
            if is_author_node(G, a) and (eligible is None or a in eligible)
        ]
        authors = sorted(set(authors), key=str)
        for u, v in combinations(authors, 2):
            counts[canon_pair(u, v)] += 1
    return counts


def threshold_edges(G, threshold, eligible_authors=None):
    counts = coauthorship_counts(G, eligible_authors)
    return {pair for pair, n in counts.items() if n >= threshold}


def safe_feature(func, graph, u, v, default=np.nan):
    try:
        if u not in graph or v not in graph:
            return default
        value = func(graph, u, v)
        return default if value is None else value
    except Exception:
        return default


def safe_len_feature(func, graph, u, v, default=0):
    try:
        if u not in graph or v not in graph:
            return default
        value = func(graph, u, v)
        return default if value is None else len(value)
    except Exception:
        return default


def paper_count(G, author):
    if author not in G:
        return 0
    return sum(1 for n in G.neighbors(author) if is_article_node(G, n))


def shortest_path_safe(G, u, v):
    try:
        if u not in G or v not in G:
            return np.nan
        return nx.shortest_path_length(G, u, v)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return np.nan


def historical_hardness(G_sub, u, v):
    if u not in G_sub or v not in G_sub:
        return False
    try:
        if len(list(nx.common_neighbors(G_sub, u, v))) > 0:
            return True
    except Exception:
        pass
    try:
        return nx.shortest_path_length(G_sub, u, v) <= 3
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return False


def build_forecast_pairs(
    G_history,
    G_future,
    S_history,
    threshold=1,
    negative_ratio=1,
    negative_mode='random',
    max_positives=None,
    seed=41
):
    rng = random.Random(seed)

    hist_authors = author_nodes(G_history)
    future_authors = author_nodes(G_future)
    structural_authors = set(S_history.nodes())

    # Cohorte temporal: autores conocidos antes del horizonte y observados
    # en el horizonte futuro.
    # NO se exige pertenecer al subgrafo histórico: no tener enlaces
    # estructurales previos es información legítima del estado histórico
    # y no debe excluir al candidato.
    eligible = sorted(
        hist_authors.intersection(future_authors),
        key=str,
    )

    if len(eligible) < 2:
        raise ValueError(
            'No hay suficientes autores elegibles para construir pares.'
        )

    # Seguridad: ningún candidato puede ser keyword, afiliación o artículo.
    bad_hist = [n for n in eligible if not is_author_node(G_history, n)]
    bad_future = [n for n in eligible if not is_author_node(G_future, n)]

    if bad_hist or bad_future:
        raise AssertionError(
            'La cohorte contiene nodos que no son autores.'
        )

    # Enlaces históricos y enlaces observados en el horizonte futuro.
    hist_positive = threshold_edges(
        G_history,
        threshold,
        eligible_authors=eligible
    )

    future_positive_all = threshold_edges(
        G_future,
        threshold,
        eligible_authors=eligible
    )

    # Nuevas colaboraciones: aparecen en el horizonte futuro
    # y no existían históricamente.
    new_future_positive_all = future_positive_all - hist_positive

    # Universo natural de candidatos.
    total_unordered_pairs = len(eligible) * (len(eligible) - 1) // 2
    candidate_pairs = total_unordered_pairs - len(hist_positive)

    natural_positive_pairs = len(new_future_positive_all)
    natural_negative_pairs = candidate_pairs - natural_positive_pairs

    natural_positive_prevalence = (
        natural_positive_pairs / candidate_pairs
        if candidate_pairs > 0
        else np.nan
    )

    # Positivos que se utilizarán en la muestra experimental.
    future_positive = sorted(
        new_future_positive_all,
        key=lambda x: (x[0], x[1])
    )

    if max_positives is not None and len(future_positive) > max_positives:
        future_positive = rng.sample(
            future_positive,
            max_positives
        )

    positives = set(future_positive)

    n_neg = len(positives) * int(negative_ratio)

    if n_neg == 0:
        raise ValueError(
            'No se encontraron positivos nuevos con el umbral elegido.'
        )

    negatives = set()
    attempts = 0
    max_attempts = max(100000, n_neg * 500)

    while len(negatives) < n_neg and attempts < max_attempts:
        attempts += 1

        u, v = rng.sample(eligible, 2)
        pair = canon_pair(u, v)

        # Un negativo no puede:
        # 1) ser colaboración histórica,
        # 2) convertirse en colaboración durante el horizonte futuro,
        # 3) haber sido seleccionado previamente como negativo.
        if (
            pair in hist_positive
            or pair in new_future_positive_all
            or pair in negatives
        ):
            continue

        if (
            negative_mode == 'hard'
            and not historical_hardness(S_history, u, v)
        ):
            continue

        negatives.add(pair)

    if len(negatives) < n_neg:
        raise RuntimeError(
            f'No fue posible obtener {n_neg} negativos '
            f'{negative_mode}; se obtuvieron {len(negatives)}.'
        )

    rows = (
        [(u, v, 1) for u, v in positives]
        + [(u, v, 0) for u, v in negatives]
    )

    df = pd.DataFrame(
        rows,
        columns=['source', 'target', 'connected']
    )

    df = df.sample(
        frac=1,
        random_state=seed
    ).reset_index(drop=True)

    audit = {
        'historical_authors': len(hist_authors),
        'future_authors': len(future_authors),
        'historical_structural_nodes': len(structural_authors),
        'historical_structural_active_authors':
            sum(
                1 for n in S_history.nodes
                if S_history.degree(n) > 0
            ),
        'eligible_authors': len(eligible),
        'eligible_with_structural_links':
            sum(
                1 for n in eligible
                if n in S_history and S_history.degree(n) > 0
            ),
        'eligible_without_structural_links':
            sum(
                1 for n in eligible
                if n not in S_history or S_history.degree(n) == 0
            ),
        'historical_positive_pairs': len(hist_positive),
        'future_positive_pairs_all': len(future_positive_all),

        # Número REAL de nuevos positivos en el universo candidato.
        'new_future_positive_pairs_all':
            len(new_future_positive_all),

        # Número de positivos efectivamente incluidos en la muestra.
        'sampled_positive_pairs': len(positives),

        'negative_pairs': len(negatives),
        'negative_ratio': negative_ratio,
        'negative_mode': negative_mode,
        'threshold': threshold,

        # Sparsity / prevalencia natural.
        'total_unordered_pairs': total_unordered_pairs,
        'candidate_pairs': candidate_pairs,
        'natural_positive_pairs': natural_positive_pairs,
        'natural_negative_pairs': natural_negative_pairs,
        'natural_positive_prevalence':
            natural_positive_prevalence,
    }

    return df, audit


def add_historical_features(df, G_history, S_history):
    out = df.copy()

    # Todos los autores históricos deben existir en S_history; quienes no tenían
    # coautorías previas permanecen como nodos aislados.
    missing_structural = (set(out['source']) | set(out['target'])) - set(S_history.nodes())
    if missing_structural:
        raise AssertionError(f'Faltan {len(missing_structural)} autores históricos en S_history.')
    isolated_candidates = {n for n in (set(out['source']) | set(out['target'])) if S_history.degree(n) == 0}
    print(f'Autores candidatos estructuralmente aislados: {len(isolated_candidates):,}')

    # FP
    out['sum_of_papers'] = out.apply(lambda r: paper_count(G_history, r['source']) + paper_count(G_history, r['target']), axis=1)
    out['sum_of_keywords'] = out.apply(lambda r: safe_len_feature(ft.sum_keywords, G_history, r['source'], r['target'], 0), axis=1)
    out['keywords_match'] = out.apply(lambda r: safe_len_feature(ft.keyword_match, G_history, r['source'], r['target'], 0), axis=1)
    out['sum_of_areas'] = out.apply(lambda r: safe_len_feature(ft.sum_areas, G_history, r['source'], r['target'], 0), axis=1)

    # LSP
    out['sum_of_neighbors'] = out.apply(lambda r: len(ft.neighbors(G_history, r['source'])) + len(ft.neighbors(G_history, r['target'])) if r['source'] in G_history and r['target'] in G_history else 0, axis=1)
    out['log_secundary_neighbors'] = out.apply(lambda r: safe_feature(ft.sum_sec_neig, G_history, r['source'], r['target'], 0.0), axis=1)
    out['lenght_short_path'] = out.apply(lambda r: shortest_path_safe(S_history, r['source'], r['target']), axis=1)
    out['clustering_index_sum'] = out.apply(lambda r: safe_feature(ft.clustering_index, S_history, r['source'], r['target'], np.nan), axis=1)
    out['jaccard_coefficient'] = out.apply(lambda r: safe_feature(ft.jaccard_coefficient, S_history, r['source'], r['target'], 0.0), axis=1)
    out['resource_allocation'] = out.apply(lambda r: safe_feature(ft.get_resource_allocation_index, S_history, r['source'], r['target'], 0.0), axis=1)
    out['adamic_adar_index'] = out.apply(lambda r: safe_feature(ft.get_adamic_adar_index, S_history, r['source'], r['target'], 0.0), axis=1)
    out['preferential_attachment'] = out.apply(lambda r: safe_feature(ft.get_preferential_attachment, S_history, r['source'], r['target'], 0.0), axis=1)

    # GSP
    out['community_cn'] = out.apply(lambda r: safe_feature(ft.community_cn, S_history, r['source'], r['target'], 0.0), axis=1)
    out['community_ra'] = out.apply(lambda r: safe_feature(ft.community_ra, S_history, r['source'], r['target'], 0.0), axis=1)
    return out


# Representación estructural histórica completa: autores conocidos sin coautorías previas
# se incorporan como nodos aislados, sin añadir aristas.
S_2000_2020_RAW = S_2000_2020.copy()
S_HIST_2022_RAW = S_HIST_2022.copy()
S_2000_2020, ISOLATED_2000_2020 = add_isolated_historical_authors(S_2000_2020_RAW, G_2000_2020)
S_HIST_2022, ISOLATED_2000_2022 = add_isolated_historical_authors(S_HIST_2022_RAW, G_HIST_2022)

print(f'Autores históricos añadidos como aislados 2000-2020: {len(ISOLATED_2000_2020):,}')
print(f'Autores históricos añadidos como aislados 2000-2022: {len(ISOLATED_2000_2022):,}')

# TRAIN/VAL: historia 2000-2020 -> futuro 2021-2022
forecast_21_22, audit_21_22 = build_forecast_pairs(
    G_2000_2020, G_2021_2022, S_2000_2020,
    threshold=MIN_COAUTHORED_PAPERS,
    negative_ratio=NEGATIVE_RATIO,
    negative_mode=NEGATIVE_MODE,
    max_positives=MAX_POSITIVES_TRAINVAL,
    seed=SEED,
)
forecast_21_22 = add_historical_features(forecast_21_22, G_2000_2020, S_2000_2020)

# Comprobacion antes de dividir: cada clase debe tener al menos 2 casos para stratify.
_class_counts = forecast_21_22['connected'].value_counts()
print('TRAIN/VAL antes de dividir:', _class_counts.to_dict())
if len(_class_counts) < 2 or _class_counts.min() < 2:
    raise ValueError(
        f'No hay suficientes casos por clase para train/val: {_class_counts.to_dict()}. '
        'Revise el umbral o la construccion de candidatos.'
    )

sample_train, sample_val = train_test_split(
    forecast_21_22,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=forecast_21_22['connected'],
)
sample_train = sample_train.reset_index(drop=True)
sample_val = sample_val.reset_index(drop=True)

# TEST: historia 2000-2022 -> futuro 2023-2024
sample_test, audit_23_24 = build_forecast_pairs(
    G_HIST_2022, G_2023_2024, S_HIST_2022,
    threshold=MIN_COAUTHORED_PAPERS,
    negative_ratio=NEGATIVE_RATIO,
    negative_mode=NEGATIVE_MODE,
    max_positives=MAX_POSITIVES_TEST,
    seed=SEED + 1,
)
sample_test = add_historical_features(sample_test, G_HIST_2022, S_HIST_2022)


def summarize(name, df):
    print(f'\n{name}')
    print(f'filas: {len(df):,}')
    print(df['connected'].value_counts().sort_index())
    print(f'proporción positiva: {df["connected"].mean():.4f}')
    print(f'duplicados: {df.duplicated(["source", "target"]).sum()}')
    nas = df.isna().sum()
    print('NaN:')
    print(nas[nas > 0])
    for col in [
        'lenght_short_path', 'clustering_index_sum', 'jaccard_coefficient',
        'resource_allocation', 'adamic_adar_index',
        'preferential_attachment', 'community_cn', 'community_ra'
    ]:
        if col in df.columns:
            non_null = int(df[col].notna().sum())
            non_zero = int((df[col].fillna(0) != 0).sum())
            print(f'{col}: no-NaN={non_null}/{len(df)}, no-cero={non_zero}/{len(df)}')


summarize('TRAIN', sample_train)
summarize('VAL', sample_val)
summarize('TEST', sample_test)

pairs_train = set(map(tuple, sample_train[['source', 'target']].to_numpy()))
pairs_val = set(map(tuple, sample_val[['source', 'target']].to_numpy()))
pairs_test = set(map(tuple, sample_test[['source', 'target']].to_numpy()))
assert pairs_train.isdisjoint(pairs_val), 'Hay pares repetidos entre train y val.'
assert pairs_train.isdisjoint(pairs_test), 'Hay pares repetidos entre train y test.'
assert pairs_val.isdisjoint(pairs_test), 'Hay pares repetidos entre val y test.'


FEATURE_COLUMNS = [
    'sum_of_papers', 'sum_of_neighbors', 'log_secundary_neighbors',
    'sum_of_keywords', 'keywords_match', 'sum_of_areas',
    'lenght_short_path', 'clustering_index_sum', 'jaccard_coefficient',
    'resource_allocation', 'adamic_adar_index', 'preferential_attachment',
    'community_cn', 'community_ra'
]
required = {'source', 'target', 'connected', *FEATURE_COLUMNS}
for name, df in [('train', sample_train), ('val', sample_val), ('test', sample_test)]:
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise AssertionError(f'{name}: faltan columnas {missing_cols}')

suffix = f'thr{MIN_COAUTHORED_PAPERS}_neg{NEGATIVE_RATIO}_{NEGATIVE_MODE}_isolates'
train_file = BASE_DIR / f'sample_temporal_train_hist2000_2020_target2021_2022_{suffix}.csv'
val_file = BASE_DIR / f'sample_temporal_val_hist2000_2020_target2021_2022_{suffix}.csv'
test_file = BASE_DIR / f'sample_temporal_test_hist2000_2022_target2023_2024_{suffix}.csv'

sample_train.to_csv(train_file, index=False)
sample_val.to_csv(val_file, index=False)
sample_test.to_csv(test_file, index=False)

print('\nArchivos generados:')
print(train_file)
print(val_file)
print(test_file)

audit_df = pd.DataFrame([
    {'split': 'train_val', 'feature_history': '2000-2020', 'label_horizon': '2021-2022', **audit_21_22},
    {'split': 'test', 'feature_history': '2000-2022', 'label_horizon': '2023-2024', **audit_23_24},
])
audit_file = BASE_DIR / f'temporal_sampling_audit_{suffix}.csv'
audit_df.to_csv(audit_file, index=False)
print(audit_file)

if WRITE_LEGACY_ALIASES:
    sample_train.to_csv(BASE_DIR / 'sample_train_2000_2020.csv', index=False)
    sample_val.to_csv(BASE_DIR / 'sample_val_2021_2022.csv', index=False)
    sample_test.to_csv(BASE_DIR / 'sample_test_2023_2024.csv', index=False)
    print('También se escribieron alias legacy.')

# Configuraciones recomendadas para responder revisores:
# 1) Principal (coherente con P4 original): threshold=1, ratio=1, random
# 2) Desbalance: threshold=1, ratio=10, random
# 3) Hard negatives: threshold=1, ratio=10, hard
# 4) Sensibilidad: threshold=1,2,3,5


Grafos cargados:
G_2000_2020: nodos=9,041, aristas=14,064
G_2021_2022: nodos=5,151, aristas=6,752
G_2023_2024: nodos=3,207, aristas=4,025
G_HIST_2022: nodos=13,371, aristas=20,728
Autores históricos añadidos como aislados 2000-2020: 0
Autores históricos añadidos como aislados 2000-2022: 0
Autores candidatos estructuralmente aislados: 6
TRAIN/VAL antes de dividir: {1: 17, 0: 17}
Autores candidatos estructuralmente aislados: 21

TRAIN
filas: 27
connected
0    13
1    14
Name: count, dtype: int64
proporción positiva: 0.5185
duplicados: 0
NaN:
lenght_short_path    13
dtype: int64
lenght_short_path: no-NaN=14/27, no-cero=14/27
clustering_index_sum: no-NaN=27/27, no-cero=24/27
jaccard_coefficient: no-NaN=27/27, no-cero=7/27
resource_allocation: no-NaN=27/27, no-cero=7/27
adamic_adar_index: no-NaN=27/27, no-cero=7/27
preferential_attachment: no-NaN=27/27, no-cero=18/27
community_cn: no-NaN=27/27, no-cero=7/27
community_ra: no-NaN=27/27, no-cero=1/27

VAL
filas: 7
connected
0    4
1    3
Name: